# Grammar Scoring Engine for Spoken English
### SHL Hiring Assessment 2026

**Task.** Predict a continuous grammar score (0–5) for a 45–60 s recording of spoken English. Labels are MOS Likert grammar scores following the 1–5 rubric. The leaderboard combines Pearson correlation and RMSE.

## Report

### What the data showed (and what was done about it)

| finding | consequence | fix |
|---|---|---|
| The 37 clips labelled 0.0 are pure synthetic white noise (spectral flatness ≈ 0.53, speech ≈ 0.01; file IDs 5037–5073, outside the real range). No test clip is noise. | They would pull predictions toward 0 and flatter the CV score. | Removed from training; a simple noise rule still predicts 0 for noise-only input. |
| Test clips are shorter than training clips (median 45 s vs 60 s). | Anything that counts seconds or words, and even the embeddings, would differ between train and test for reasons that have nothing to do with grammar. | Count features are replaced by rates; every training clip is cut into 4 pieces whose lengths follow the test-length distribution (section 3b). |
| Many training clips come from the same speaker, while test speakers are new. | A model can recognise a voice it has heard in training, so ordinary CV looks better than the test score. | Speaker groups from ECAPA voice embeddings; all clips of a speaker stay in one CV fold (section 6a). |

### Pipeline

1. **Clean the data** and cut each training clip into 4 test-length crops (2,928 training rows from 732 clips).
2. **Features computed for every crop and test clip:**
   - audio quality (loudness, noise floor, SNR, spectral flatness);
   - fluency from Whisper `small.en` word timestamps (speech rate, pauses, ASR confidence);
   - grammar from the transcript: sentence acceptability (RoBERTa fine-tuned on CoLA), how much a T5 grammar-correction model has to change, ERRANT error types, part-of-speech changes;
   - embeddings: MPNet sentence embedding of the transcript, WavLM base+ and WavLM-large, and the encoders of Whisper `small.en`, `medium.en` and `large-v3`.
3. **Text models:** every clip is transcribed again with Whisper `large-v3-turbo` (better recognition), and DeBERTa-v3 base and large are fine-tuned to predict the score directly from the transcript.
4. **Base models:** Ridge, SVR and LightGBM on all features, one Ridge per embedding block, kernel ridge, k-nearest neighbours, and the two DeBERTa models.
5. **Stacking:** a non-negative linear regression combines the out-of-fold predictions of the base models. Predictions are clipped to [1, 5].

All models are evaluated with the same 5-fold split, stratified by score and grouped by speaker, and every crop of a clip stays in its clip's fold.

### Results

| | CV RMSE | CV Pearson | public leaderboard |
|---|---|---|---|
| final stacked ensemble | **0.532** | **0.851** | **0.3465** |
| best single model (SVR) | 0.564 | 0.832 | |
| DeBERTa-v3-large on transcripts | 0.610 | 0.800 | |
| predict the mean | 1.014 | – | |

The compulsory **training RMSE** of the final ensemble is printed in section 7. It is much lower than the CV RMSE because the models have seen those labels; CV RMSE is the honest estimate of test performance.

### How the score improved (public leaderboard, lower is better)

| version | change | leaderboard |
|---|---|---|
| v4 | hand-made features + speech-model embeddings, stacked; ordinary 5-fold CV | 0.4139 |
| v11 | removed length-dependent features; speaker-grouped CV | 0.3944 |
| v12 | trained on test-length crops | 0.3905 |
| v14 | part-of-speech features, WavLM-large, ECAPA speaker groups | 0.3897 |
| v15 | Whisper `large-v3-turbo` transcripts + fine-tuned DeBERTa-v3-base | 0.3502 |
| v16 | + DeBERTa-v3-large (this notebook) | **0.3465** |

CV numbers are not comparable across versions, because the folds became stricter (speaker groups, crops) along the way.

### Tried, did not help

- pseudo-labelling the test transcripts with the ensemble's own predictions (leaderboard 0.3556);
- an even stricter speaker threshold for the folds (0.3492);
- more text models: ELECTRA-large, DeBERTa on the `small.en` transcripts (0.3486);
- fine-tuning WavLM-large or Whisper `small.en` end to end on the audio (fold-0 RMSE 0.686 / 0.677; WavLM's mistakes correlate 0.86 with the frozen-embedding Ridge);
- rating transcripts against the rubric with a 7B instruction-tuned LLM (correlation 0.50 with the labels on its own).

**How to run.** On Kaggle, set Accelerator = GPU T4 and Internet = On, add the competition data, then Run All. A first run takes about 5 hours (mostly transcription, encoders and fine-tuning); every slow step is cached in the working directory, so later runs take about 1.5 hours. Set `RUN_DEEP = False` for a quick CPU-only baseline that uses only the audio-quality features.

## 1. Setup

In [ ]:
!pip install -q openai-whisper sentence-transformers lightgbm

In [ ]:
import os, re, glob, wave, difflib, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.svm import SVR
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)

RUN_DEEP = os.environ.get("RUN_DEEP", "1") == "1"  # False = quick CPU baseline on audio-quality features only
WHISPER_MODEL = "small.en"                         # transcription; "medium.en" transcribes better but is ~2x slower
ENCODERS = {"small.en": "ws_", "medium.en": "wm_", "large-v3": "wl_"}  # Whisper encoders used for embeddings -> column prefix

# Works on Kaggle (/kaggle/input/...) and locally (notebook next to the Dataset_Final folder).
DATA_DIR = Path(next(p for p in glob.glob("/kaggle/input/**/train.csv", recursive=True) + ["Dataset_Final/train.csv"]
                     if os.path.exists(p))).parent
WORK_DIR = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path("work")
WORK_DIR.mkdir(exist_ok=True)
print(f"data: {DATA_DIR} | cache: {WORK_DIR} | deep features: {RUN_DEEP}")


def cached(name, fn):
    """Run fn() once and pickle the result, so re-running the notebook skips slow steps."""
    path = WORK_DIR / f"{name}.pkl"
    if path.exists():
        return pd.read_pickle(path)
    out = fn()
    pd.to_pickle(out, path)
    return out

## 2. Data and exploration

`sample_submission.csv` is stale: only 25 of its 204 file names appear in `test.csv`. The submission is therefore built from `test.csv`, which has 216 rows.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train["path"] = [str(DATA_DIR / "train" / f) for f in train.filename]
test["path"] = [str(DATA_DIR / "test" / f) for f in test.filename]
print("train:", train.shape, "| test:", test.shape)
print(train.label.value_counts().sort_index().to_string())

In [ ]:
def load_wav(path):
    """All clips are 16 kHz mono 16-bit PCM, so the standard-library wave module is enough.

    A path of the form "clip.wav|start|end" (seconds) loads only that part; section 3b uses it for training crops.
    """
    path, *span = path.split("|")
    with wave.open(path) as w:
        assert w.getframerate() == 16000 and w.getnchannels() == 1, path
        x = np.frombuffer(w.readframes(w.getnframes()), np.int16).astype(np.float32) / 32768
    return x[int(float(span[0]) * 16000):int(float(span[1]) * 16000)] if span else x


def audio_quality(x, sr=16000):
    """Clip-level loudness and noise statistics from 25 ms frames with a 10 ms hop."""
    n, hop = 400, 160
    frames = np.lib.stride_tricks.sliding_window_view(x, n)[::hop] * np.hanning(n)
    power = np.abs(np.fft.rfft(frames, axis=1)) ** 2 + 1e-12
    frame_db = 10 * np.log10(power.sum(1))
    flatness = np.exp(np.log(power).mean(1)) / power.mean(1)  # ~0.55 for white noise, ~0.01 for speech
    return {
        "ac_duration": len(x) / sr,
        "ac_rms_db": 20 * np.log10(np.sqrt((x ** 2).mean()) + 1e-9),
        "ac_floor_db": np.percentile(frame_db, 10),
        "ac_snr_db": np.percentile(frame_db, 90) - np.percentile(frame_db, 10),
        "ac_flatness": np.median(flatness),
        "ac_zcr": (np.diff(np.sign(x)) != 0).mean(),
    }


def audio_quality_table(df):
    return pd.DataFrame([audio_quality(load_wav(p)) for p in df.path], index=df.filename)


def is_noise(ac):
    """Noise-only audio: flat spectrum and almost no loud/quiet contrast."""
    return (ac.ac_flatness > 0.3) & (ac.ac_snr_db < 10)


ac_train = cached("ac_train", lambda: audio_quality_table(train))
ac_test = cached("ac_test", lambda: audio_quality_table(test))

noise = is_noise(ac_train.loc[train.filename]).values
print(f"noise clips in train: {noise.sum()} | label-0 clips: {(train.label == 0).sum()} | "
      f"noise clips in test: {is_noise(ac_test).sum()}")
assert (noise == (train.label == 0).values).all(), "noise rule should match the label-0 clips exactly"

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4))
train.label.value_counts().sort_index().plot.bar(ax=ax[0], title="Label distribution (train)")
ax[1].hist([ac_train.ac_duration, ac_test.ac_duration], bins=30, label=["train", "test"])
ax[1].set(title="Clip duration (s): test clips are shorter")
ax[1].legend()
ax[2].scatter(ac_train.loc[train.filename].ac_snr_db, ac_train.loc[train.filename].ac_flatness,
              c=np.where(noise, "red", "steelblue"), s=10)
ax[2].set(xlabel="SNR (dB)", ylabel="spectral flatness", title="Label-0 clips (red) are pure noise")
plt.tight_layout()
plt.show()

train = train[~noise].reset_index(drop=True)
print("training clips after removing noise:", len(train))

## 3. Speech-to-text and fluency features

Whisper normally "cleans up" speech. With a disfluent `initial_prompt` it keeps fillers and false starts, which are exactly the signal a grammar rater hears. `condition_on_previous_text=False` stops one segment's errors from spilling into the next.

In [ ]:
if RUN_DEEP:
    import torch
    import whisper
    from tqdm.auto import tqdm
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", DEVICE)

PROMPT = "Umm, let me think like, hmm... Okay, here's what I'm, like, thinking."


def transcribe(model, path):
    r = model.transcribe(path, language="en", word_timestamps=True, condition_on_previous_text=False,
                         initial_prompt=PROMPT, fp16=DEVICE == "cuda")
    segs = r["segments"]
    return {
        "text": r["text"].strip(),
        "words": [(w["word"].strip(), w["start"], w["end"]) for s in segs for w in s.get("words", [])],
        "avg_logprob": np.mean([s["avg_logprob"] for s in segs]) if segs else np.nan,
        "no_speech_prob": np.mean([s["no_speech_prob"] for s in segs]) if segs else np.nan,
    }


if RUN_DEEP:
    asr = whisper.load_model(WHISPER_MODEL, device=DEVICE)
    asr_train = cached(f"asr_train_{WHISPER_MODEL}",
                       lambda: {f: transcribe(asr, p) for f, p in tqdm(list(zip(train.filename, train.path)))})
    asr_test = cached(f"asr_test_{WHISPER_MODEL}",
                      lambda: {f: transcribe(asr, p) for f, p in tqdm(list(zip(test.filename, test.path)))})
    del asr
    labels = train.set_index("filename").label
    for f in train.sort_values("label").filename.iloc[[0, len(train) // 2, -1]]:
        print(f"[{labels[f]}] {asr_train[f]['text'][:300]}\n")

### 3b. Cutting the training clips to test length

Test clips are shorter than training clips. The median test clip lasts 45 s and the median training clip 60 s; 71% of test clips are 55 s or shorter, against 26% of training clips. Removing length features (section 6a) is not enough, because length also leaks into the embeddings. Whisper, for example, encodes audio in 30 s windows. A 60 s clip fills two windows, while a 45 s clip fills one and a half, and the rest is silence padding. A model trained only on full-length clips has rarely seen that.

So every training clip is cut into `N_CROPS` pieces whose lengths are drawn from the test-set length distribution. Only the test durations are used, never test labels.
- The first piece starts at 0 s, like a test recording. The others start at a random sentence start, so they do not begin mid-sentence.
- Each piece keeps its clip's score.
- Transcripts for the pieces come from Whisper's word timestamps, so no audio is transcribed twice.

Every feature in sections 3–5 is computed on these pieces, and the models are trained on them. All pieces of one clip always go to the same CV fold, so CV measures how well the model scores an unseen clip at test length. The models also get 4 times as many training rows.

In [ ]:
N_CROPS = 4


def make_crops(df, asr_out, test_durations, seed=SEED):
    """Cut every clip into N_CROPS pieces with lengths drawn from the test-set durations."""
    rng = np.random.default_rng(seed)
    rows = []
    for i, (f, p, label) in enumerate(zip(df.filename, df.path, df.label)):
        dur = ac_train.loc[f, "ac_duration"]
        words = asr_out[f]["words"] if f in asr_out else []
        sentence_starts = [0.0] + [w[1] for prev, w in zip(words, words[1:]) if prev[0].endswith((".", "!", "?"))]
        for k in range(N_CROPS):
            length = min(float(rng.choice(test_durations)), dur)
            starts = [t for t in sentence_starts if t + length <= dur] or [0.0]
            start = 0.0 if k == 0 else float(rng.choice(starts))
            rows.append({"filename": f"{f}#{k}", "parent": i, "label": label, "start": start, "end": start + length,
                         "path": f"{p}|{start:.3f}|{start + length:.3f}"})
    return pd.DataFrame(rows)


def crop_asr(a, start, end):
    """The part of a clip's transcript inside [start, end], with word times relative to the crop."""
    words = [(w, t0 - start, t1 - start) for w, t0, t1 in a["words"] if t0 >= start and t1 <= end]
    return {**a, "text": " ".join(w for w, _, _ in words), "words": words}


crops = make_crops(train, asr_train if RUN_DEEP else {}, ac_test.ac_duration.values)
if RUN_DEEP:
    asr_crops = {c: crop_asr(asr_train[f], t0, t1) for c, f, t0, t1 in
                 zip(crops.filename, train.filename.values[crops.parent], crops.start, crops.end)}
ac_crops = cached(f"ac_crops{N_CROPS}", lambda: audio_quality_table(crops))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist([crops.end - crops.start, ac_test.ac_duration], bins=30, density=True, label=["training crops", "test clips"])
ax.set(xlabel="length (s)", title="Training crops follow the test-length distribution")
ax.legend()
plt.show()
print("training rows:", len(crops), "crops from", len(train), "clips")

In [ ]:
FILLERS = {"um", "umm", "uh", "uhh", "er", "erm", "ah", "hmm", "mm"}


def fluency(a):
    """Rate, pause, filler and repetition statistics from Whisper word timestamps."""
    words = [(re.sub(r"[^\w']", "", w).lower(), s, e) for w, s, e in a["words"]]
    words = [w for w in words if w[0]]
    if len(words) < 2:
        return {"fl_words_per_sec": 0.0}
    tokens = [w for w, _, _ in words]
    gaps = np.array([nxt[1] - cur[2] for cur, nxt in zip(words, words[1:])])
    speech_time = max(words[-1][2] - words[0][1], 1e-3)
    return {
        "fl_words_per_sec": len(words) / speech_time,
        "fl_pauses_per_min": (gaps > 0.5).sum() / speech_time * 60,
        "fl_mean_pause": gaps[gaps > 0.5].mean() if (gaps > 0.5).any() else 0.0,
        "fl_filler_rate": sum(t in FILLERS for t in tokens) / len(tokens),
        "fl_repeat_rate": sum(t1 == t2 for t1, t2 in zip(tokens, tokens[1:])) / len(tokens),
        # MATTR: type-token ratio averaged over 50-word windows, so it does not shrink as clips get longer
        "fl_mattr": np.mean([len(set(tokens[i:i + 50])) / len(tokens[i:i + 50])
                             for i in range(max(1, len(tokens) - 49))]),
        "fl_avg_logprob": a["avg_logprob"],
        "fl_no_speech_prob": a["no_speech_prob"],
    }


if RUN_DEEP:
    fl_crops = pd.DataFrame.from_dict({f: fluency(a) for f, a in asr_crops.items()}, orient="index")
    fl_test = pd.DataFrame.from_dict({f: fluency(a) for f, a in asr_test.items()}, orient="index")
    print(fl_crops.describe().T.round(3))

## 4. Grammar features

All grammar features are rates or averages, not raw counts, because test clips are shorter than training clips.
- **Acceptability**: P(grammatical) for each sentence from `textattack/roberta-base-CoLA`, summarised per clip as mean, minimum and share of sentences below 0.5.
- **Correction edit rate**: share of words that `vennify/t5-base-grammar-correction` changes when it fixes each sentence. More fixes means worse grammar.

In [ ]:
def sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if len(s.split()) >= 3]


def edit_rate(orig, fixed):
    """Share of words changed between a sentence and its correction (word-level diff, punctuation ignored)."""
    a, b = re.findall(r"[\w']+", orig.lower()), re.findall(r"[\w']+", fixed.lower())
    ops = difflib.SequenceMatcher(None, a, b).get_opcodes()
    return sum(max(i2 - i1, j2 - j1) for op, i1, i2, j1, j2 in ops if op != "equal") / max(len(a), 1)


ACC_MEMO, GEC_MEMO = {}, {}  # crops repeat most sentences of their clip, so each sentence is scored only once


def acceptability(sents, bs=64):
    """P(acceptable) per sentence; label 1 means acceptable for this CoLA model."""
    todo = [s for s in dict.fromkeys(sents) if s not in ACC_MEMO]
    with torch.no_grad():
        for i in tqdm(range(0, len(todo), bs), desc="CoLA"):
            batch = todo[i:i + bs]
            enc = cola_tok(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(DEVICE)
            ACC_MEMO.update(zip(batch, cola(**enc).logits.softmax(-1)[:, 1].tolist()))
    return [ACC_MEMO[s] for s in sents]


def correct(sents, bs=32):
    todo = [s for s in dict.fromkeys(sents) if s not in GEC_MEMO]
    with torch.no_grad():
        for i in tqdm(range(0, len(todo), bs), desc="GEC"):
            batch = todo[i:i + bs]
            enc = gec_tok(["grammar: " + s for s in batch], padding=True, truncation=True,
                          max_length=128, return_tensors="pt").to(DEVICE)
            GEC_MEMO.update(zip(batch, gec_tok.batch_decode(gec.generate(**enc, max_new_tokens=128, num_beams=2),
                                                            skip_special_tokens=True)))
    return [GEC_MEMO[s] for s in sents]


def grammar_table(asr_out):
    sents = {f: sentences(a["text"]) for f, a in asr_out.items()}
    flat = [s for ss in sents.values() for s in ss]
    acc, fixed = iter(acceptability(flat)), iter(correct(flat))
    rows = {}
    for f, ss in sents.items():
        words = asr_out[f]["words"]
        minutes = max(words[-1][2] - words[0][1], 1.0) / 60 if words else np.nan  # speaking time, from word timestamps
        p = np.array([next(acc) for _ in ss])
        e = np.array([edit_rate(s, next(fixed)) for s in ss])
        rows[f] = {"gr_sent_per_min": len(ss) / minutes} if not ss else {
            "gr_sent_per_min": len(ss) / minutes,
            "gr_mean_sent_len": np.mean([len(s.split()) for s in ss]),
            "gr_accept_mean": p.mean(), "gr_accept_min": p.min(), "gr_unaccept_frac": (p < 0.5).mean(),
            "gr_edit_rate": e.mean(), "gr_edit_max": e.max(),
        }
    return pd.DataFrame.from_dict(rows, orient="index")


if RUN_DEEP:
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
    cola_tok = AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
    cola = AutoModelForSequenceClassification.from_pretrained("textattack/roberta-base-CoLA").to(DEVICE).eval()
    gec_tok = AutoTokenizer.from_pretrained("vennify/t5-base-grammar-correction")
    gec = AutoModelForSeq2SeqLM.from_pretrained("vennify/t5-base-grammar-correction").to(DEVICE).eval()

    # Sanity check: the models must rank a correct sentence above a broken one.
    good, bad = acceptability(["She went to school yesterday.", "She go to school yesterday and buyed a apple."])
    fixed = correct(["She go to school yesterday and buyed a apple."])[0]
    print(f"acceptability good={good:.2f} bad={bad:.2f} | correction: {fixed!r}")
    assert good > bad

    gr_crops = cached(f"gr2_crops{N_CROPS}_{WHISPER_MODEL}", lambda: grammar_table(asr_crops))
    gr_test = cached(f"gr2_test_{WHISPER_MODEL}", lambda: grammar_table(asr_test))
    print(gr_crops.describe().T.round(3))

### 4c. Error types (ERRANT)

The grammar-correction model rewrites each transcript into a corrected version. Above, only the share of changed words was counted. **ERRANT**, the standard tool for evaluating grammar correction, aligns the original with the correction and labels every edit with an error type: verb tense, subject-verb agreement, missing determiner, word order and so on.

Each type becomes a rate per 100 words, so every type of mistake can carry its own weight. The rubric cares more about sentence structure than about a missing "the".

In [ ]:
ERRANT_TYPES = ["VERB:TENSE", "VERB:SVA", "VERB:FORM", "VERB", "DET", "PREP", "NOUN:NUM", "NOUN", "PRON",
                "WO", "MORPH", "ADJ", "ADV", "CONJ", "PART", "PUNCT", "ORTH", "SPELL", "OTHER"]


ERR_MEMO = {}  # sentence -> (number of tokens, ERRANT edit types)


def errant_table(asr_out):
    """Per-clip ERRANT error rates per 100 words, split into missing (M), unnecessary (U) and replacement (R) edits."""
    rows = {}
    files = list(asr_out)
    sents = {f: sentences(asr_out[f]["text"]) for f in files}
    flat = [x for f in files for x in sents[f]]
    fixed = iter(correct(flat))
    for f in tqdm(files, desc="ERRANT"):
        counts = dict.fromkeys(["er_M", "er_U", "er_R"] + [f"er_{t}" for t in ERRANT_TYPES], 0)
        n_words = 0
        for orig in sents[f]:
            cor = next(fixed)
            if orig not in ERR_MEMO:
                o = annotator.parse(orig)
                ERR_MEMO[orig] = (len(o), [e.type for e in annotator.annotate(o, annotator.parse(cor))])
            n, types = ERR_MEMO[orig]
            n_words += n
            for t in types:
                op, kind = t.split(":", 1) if ":" in t else (t, "OTHER")
                counts[f"er_{op}"] = counts.get(f"er_{op}", 0) + 1
                key = f"er_{kind}" if f"er_{kind}" in counts else f"er_{kind.split(':')[0]}"
                counts[key if key in counts else "er_OTHER"] += 1
        rows[f] = {k: 100 * v / max(n_words, 1) for k, v in counts.items()}
        rows[f]["er_total"] = sum(rows[f][k] for k in ("er_M", "er_U", "er_R"))
    return pd.DataFrame.from_dict(rows, orient="index")


if RUN_DEEP:
    import subprocess
    subprocess.run("pip install -q errant && python -m spacy download en_core_web_sm", shell=True, check=True)
    import errant
    annotator = errant.load("en")
    er_crops = cached(f"errant_crops{N_CROPS}_{WHISPER_MODEL}", lambda: errant_table(asr_crops))
    er_test = cached(f"errant_test_{WHISPER_MODEL}", lambda: errant_table(asr_test))
    er_crops.columns = er_crops.columns.str.replace(":", "_")  # LightGBM rejects ":" in feature names
    er_test.columns = er_test.columns.str.replace(":", "_")
    er_crops = er_crops.loc[:, er_crops.std() > 0]  # drop error types that never occur
    er_test = er_test.reindex(columns=er_crops.columns, fill_value=0.0)
    corr = er_crops.loc[crops.filename].corrwith(crops.set_index("filename").label).sort_values()
    print("Correlation of each error rate with the human score (most negative = most harmful):")
    print(corr.round(3).to_string())

### 4d. Grammar change and grammar range (part-of-speech tags)

ERRANT says *what kind* of edit the correction model made. Two more views come from spaCy part-of-speech tags (the `en_core_web_sm` model ERRANT already loads):
- **Grammar change.** Each sentence and its correction are tagged, and the tag counts are compared, for example how many past-tense verbs (`VBD`), modals (`MD`) or determiners (`DT`) the correction had to add or remove. Features are `1 − cosine` between the original and corrected tag vectors over the changed sentences only, and the per-tag difference (corrected − original) per 100 words.
- **Grammar range.** The rubric gives level 5 to speakers who handle *complex* structures. From the original transcript only: subordinate connectors (because, although, which, that, when, if), perfect tenses, modals and passives, each per 100 words.

Both groups are kept only if Ridge cross-validation improves with them (section 6b-2).

In [ ]:
from collections import Counter

TAGS = ["VB", "VBD", "VBG", "VBN", "VBP", "VBZ", "MD", "PRP", "PRP$", "CC", "IN", "DT", "NN", "NNS", "JJ", "RB"]
SUBORDINATORS = {"because", "although", "which", "that", "when", "if"}


def tag_counts(doc):
    c = Counter(t.tag_ for t in doc)
    return np.array([c[t] for t in TAGS], float)


def pos_table(asr_out):
    """Per clip: POS tag change between transcript and correction, and grammatical-complexity rates per 100 words."""
    files = list(asr_out)
    sents = {f: sentences(asr_out[f]["text"]) for f in files}
    flat = list(dict.fromkeys(x for f in files for x in sents[f]))
    fixed = dict(zip(flat, correct(flat)))
    docs = dict(zip(flat, annotator.nlp.pipe(flat, batch_size=256)))
    fixed_docs = dict(zip(flat, annotator.nlp.pipe([fixed[x] for x in flat], batch_size=256)))
    rows = {}
    for f in tqdm(files, desc="POS"):
        orig, cor, orig_ch, cor_ch = (np.zeros(len(TAGS)) for _ in range(4))
        cx = dict.fromkeys(["cx_subord", "cx_perfect", "cx_modal", "cx_passive"], 0)
        n_words = 0
        for x in sents[f]:
            d = docs[x]
            o, c = tag_counts(d), tag_counts(fixed_docs[x])
            orig, cor = orig + o, cor + c
            if edit_rate(x, fixed[x]) > 0:  # only sentences the correction model changed
                orig_ch, cor_ch = orig_ch + o, cor_ch + c
            n_words += sum(not t.is_punct for t in d)
            # "that" counts only as a connector or relative pronoun (IN / WDT), not as a determiner
            cx["cx_subord"] += sum(t.lower_ in SUBORDINATORS and t.tag_ in ("IN", "WDT", "WRB") for t in d)
            cx["cx_perfect"] += sum(t.tag_ == "VBN" and any(ch.lemma_ == "have" and ch.dep_ == "aux" for ch in t.children)
                                    for t in d)
            cx["cx_modal"] += sum(t.tag_ == "MD" for t in d)
            cx["cx_passive"] += sum(t.dep_ == "auxpass" for t in d)
        k = 100 / max(n_words, 1)
        cos = orig_ch @ cor_ch / (np.linalg.norm(orig_ch) * np.linalg.norm(cor_ch) + 1e-9)
        rows[f] = {"td_tag_change": 1 - cos if orig_ch.any() else 0.0,
                   **{f"td_{t.replace('$', 'S')}": v * k for t, v in zip(TAGS, cor - orig)},
                   **{key: v * k for key, v in cx.items()}}
    return pd.DataFrame.from_dict(rows, orient="index")


if RUN_DEEP:
    pos_crops = cached(f"pos_crops{N_CROPS}_{WHISPER_MODEL}", lambda: pos_table(asr_crops))
    pos_test = cached(f"pos_test_{WHISPER_MODEL}", lambda: pos_table(asr_test)).reindex(columns=pos_crops.columns)
    corr = pos_crops.loc[crops.filename].corrwith(crops.set_index("filename").label).sort_values()
    print("Correlation of each POS feature with the human score:")
    print(corr.round(3).to_string())

## 5. Embeddings (text, WavLM layers, Whisper encoder layers)

WavLM-large is added next to WavLM-base. Its layers are pooled with the mean *and* the standard deviation over time, so the embedding also describes how much the speech varies within a clip (hesitations, restarts, uneven pace), not only its average.

In [ ]:
def text_embeddings(asr_out):
    files = list(asr_out)
    emb = sbert.encode([asr_out[f]["text"] for f in files], batch_size=32, show_progress_bar=True)
    return pd.DataFrame(emb, index=files).add_prefix("temb_")


def wavlm_layers(df, model=None, fe=None, std=False):
    """Time-averaged output of every WavLM layer (n_layers x d per clip); the best layers are chosen in section 6b.

    With std=True the standard deviation over time is appended (mean + std pooling), which also describes how
    much the speech varies within the clip: hesitations, restarts, uneven pace.
    """
    model = wavlm if model is None else model
    fe = wavlm_fe if fe is None else fe
    rows = {}
    with torch.no_grad():
        for f, p in tqdm(list(zip(df.filename, df.path)), desc="WavLM"):
            inp = fe(load_wav(p), sampling_rate=16000, return_tensors="pt").input_values.to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                hidden = model(inp, output_hidden_states=True).hidden_states
            h = torch.stack([x[0].float() for x in hidden])  # layers x frames x d
            pooled = torch.cat([h.mean(1), h.std(1)], -1) if std else h.mean(1)
            rows[f] = pooled.cpu().numpy().astype(np.float16 if std else np.float32)
    return rows


def whisper_layers(model, df):
    """Time-averaged output of every Whisper encoder block (n_layers x d per clip, stored as float16).

    The clip is encoded in 30 s windows and padding frames are dropped; the best block range is chosen in section 6b.
    """
    rows, win, outs = {}, 30 * 16000, []
    hooks = [b.register_forward_hook(lambda m, i, o: outs.append(o[0])) for b in model.encoder.blocks]
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    try:
        with torch.no_grad():
            for f, p in tqdm(list(zip(df.filename, df.path)), desc="Whisper layers"):
                x, total, frames = load_wav(p), 0, 0
                for i in range(0, len(x), win):
                    chunk = x[i:i + win]
                    if len(chunk) < 16000 and frames:  # skip a tail shorter than 1 s
                        continue
                    audio = whisper.pad_or_trim(torch.from_numpy(chunk.copy()))
                    mel = whisper.log_mel_spectrogram(audio, n_mels=model.dims.n_mels).to(DEVICE, dtype)
                    outs.clear()
                    model.embed_audio(mel[None])
                    k = max(1, int(len(chunk) / 16000 * 50))  # the encoder emits 50 frames per second
                    total = total + torch.stack([o[:k].float().sum(0) for o in outs])
                    frames += k
                rows[f] = (total / frames).cpu().numpy().astype(np.float16)
    finally:
        for h in hooks:
            h.remove()
    return rows


if RUN_DEEP:
    from sentence_transformers import SentenceTransformer
    from transformers import AutoFeatureExtractor, WavLMModel
    sbert = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device=DEVICE)
    wavlm_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus")
    wavlm = WavLMModel.from_pretrained("microsoft/wavlm-base-plus").to(DEVICE).eval()
    temb_crops = cached(f"temb_crops{N_CROPS}_{WHISPER_MODEL}", lambda: text_embeddings(asr_crops))
    temb_test = cached(f"temb_test_{WHISPER_MODEL}", lambda: text_embeddings(asr_test))
    aemb_layers_train = cached("wavlm_layers_train", lambda: wavlm_layers(train))
    aemb_layers_test = cached("wavlm_layers_test", lambda: wavlm_layers(test))
    aemb_layers_crops = cached(f"wavlm_layers_crops{N_CROPS}", lambda: wavlm_layers(crops))
    # WavLM-large (24 layers x 1024), mean + std pooled
    wavlmL_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-large")
    done = all((WORK_DIR / f"wavlmL_layers_{split}.pkl").exists() for split in (f"crops{N_CROPS}", "test"))
    wavlmL = None if done else WavLMModel.from_pretrained("microsoft/wavlm-large").to(DEVICE).eval()
    aembL_layers_crops = cached(f"wavlmL_layers_crops{N_CROPS}", lambda: wavlm_layers(crops, wavlmL, wavlmL_fe, std=True))
    aembL_layers_test = cached("wavlmL_layers_test", lambda: wavlm_layers(test, wavlmL, wavlmL_fe, std=True))
    del wavlmL
    torch.cuda.empty_cache()
    WHISPER_LAYERS = {}
    for name in ENCODERS:
        done = all((WORK_DIR / f"wlayers_{split}_{name}.pkl").exists() for split in (f"crops{N_CROPS}", "test"))
        enc_model = None if done else whisper.load_model(name, device=DEVICE)
        WHISPER_LAYERS[name] = (cached(f"wlayers_crops{N_CROPS}_{name}", lambda: whisper_layers(enc_model, crops)),
                                cached(f"wlayers_test_{name}", lambda: whisper_layers(enc_model, test)))
        del enc_model
        torch.cuda.empty_cache()

## 6. Feature table and models

Every clip becomes one row. Hand-made features are median-imputed and scaled. Each embedding block is scaled and reduced with PCA (size chosen in 6b). All of this sits inside the sklearn pipeline, so it is re-fitted on the training part of every fold.

The folds are stratified on half-point score bins, with the 4 clips below 2.0 merged into the 2.0 bin. The same 5 folds are used for every model, which makes the out-of-fold predictions comparable and lets them be stacked.

### 6a. Data checks before modelling

Better data beats a better model, so the data is checked before any model is trained:
- **Clip length.** Test clips are shorter than training clips. Any feature that counts things (seconds, words) instead of measuring a rate would mark short test clips down for their length, not for their grammar. Such features are removed, and vocabulary variety is measured as MATTR (the moving-average type-token ratio over 50-word windows), which does not depend on length. The models are also trained on test-length crops (section 3b).
- **Exact duplicates.** Identical audio files in train and test, found by hashing the raw samples.
- **Repeat speakers.** The training set holds many clips of the same speakers, while test speakers are new. Each clip gets an ECAPA speaker embedding (SpeechBrain, trained for speaker verification). Clips whose embeddings are close enough are linked, and linked clips form one speaker group. The threshold comes from the test set: almost no test clip (99%) is as close to any training clip as two recordings of one training speaker are to each other, so training pairs above that similarity are treated as the same speaker. Clips linked this way score much more alike than random pairs, which confirms they are the same person. If one clip of a speaker were in a training fold and another in the validation fold, the model could recognise the voice and CV would look better than it is, so all clips of a speaker group stay in the same fold.
- **Label disagreement.** Two recordings of the same speaker should get similar grammar scores. How far their labels differ gives a rough estimate of the label noise, a floor the RMSE cannot go below.

In [ ]:
import hashlib
from scipy.sparse.csgraph import connected_components

lab = train.set_index("filename").label
dur = ac_train.loc[train.filename].ac_duration
print(f"duration: train mean {dur.mean():.1f} s, test mean {ac_test.ac_duration.mean():.1f} s | "
      f"correlation of duration with score in train: {np.corrcoef(dur, lab.loc[train.filename])[0, 1]:.3f}")

# Exact duplicates (hash of the raw audio).
digest = {f"{split}/{f}": hashlib.md5(load_wav(p).tobytes()).hexdigest()
          for split, df in (("train", train), ("test", test)) for f, p in zip(df.filename, df.path)}
dup = pd.Series(list(digest), index=list(digest.values())).groupby(level=0).apply(list)
dup = dup[dup.map(len) > 1]
print(f"exact duplicate audio groups: {len(dup)}", dup.tolist()[:5])

def ecapa_vectors(df):
    """ECAPA-TDNN speaker embeddings (SpeechBrain, trained on VoxCeleb for speaker verification).
    20 s of audio is plenty to recognise a voice."""
    with torch.no_grad():
        return {f: ecapa.encode_batch(torch.from_numpy(load_wav(p)[:20 * 16000].copy())[None]).squeeze().cpu().numpy()
                for f, p in tqdm(list(zip(df.filename, df.path)), desc="ECAPA")}


def unit(d, files):
    E = np.stack([d[f] for f in files])
    return E / np.linalg.norm(E, axis=1, keepdims=True)


SPEAKER_GROUPS = None
if RUN_DEEP:
    import subprocess
    subprocess.run("pip install -q speechbrain", shell=True, check=True)
    try:
        from speechbrain.inference.speaker import EncoderClassifier
    except ImportError:
        from speechbrain.pretrained import EncoderClassifier
    ecapa = EncoderClassifier.from_hparams("speechbrain/spkrec-ecapa-voxceleb", savedir=str(WORK_DIR / "ecapa_model"),
                                           run_opts={"device": DEVICE})
    E = unit(cached("ecapa_train", lambda: ecapa_vectors(train)), train.filename)
    T = unit(cached("ecapa_test", lambda: ecapa_vectors(test)), test.filename)
    S = E @ E.T
    np.fill_diagonal(S, -1)
    q = [.1, .25, .5, .75, .9]
    print("nearest other training clip, cosine quantiles 10/25/50/75/90%:", np.quantile(S.max(1), q).round(3))
    test_nn = (T @ E.T).max(1)
    print("nearest training clip for each test clip:                    ", np.quantile(test_nn, q).round(3))
    # Test speakers are new: almost no test clip is closer than this to any training clip. Training pairs above it
    # are therefore the same speaker, and CV must keep them apart exactly as the test set does.
    T_MIN = np.ceil(np.quantile(test_nn, 0.99) * 20) / 20
    print(f"99% of test clips are below {np.quantile(test_nn, 0.99):.3f} -> same-speaker threshold at least {T_MIN}")
    # The threshold is the lowest one at or above T_MIN that does not chain more than 10% of the clips into one
    # group; the table shows that linked clips really score alike (much closer than random pairs).
    yl = lab.loc[train.filename].values
    random_rmse = np.sqrt(2) * yl.std()
    print(f"random-pair label RMSE {random_rmse:.3f}")
    for THRESH in np.round(np.arange(0.3, 0.86, 0.05), 2):
        n_groups, groups = connected_components(S > THRESH, directed=False)
        i, j = np.nonzero(np.triu(groups[:, None] == groups[None, :], 1))
        within = np.sqrt(((yl[i] - yl[j]) ** 2).mean()) if len(i) else np.nan
        largest = np.bincount(groups).max()
        print(f"  threshold {THRESH}: {n_groups} groups, largest {largest}, label RMSE within groups {within:.3f}")
        if SPEAKER_GROUPS is None and THRESH >= T_MIN and largest <= 0.1 * len(train):
            SPEAKER_GROUPS, CHOSEN = groups, THRESH
    if SPEAKER_GROUPS is None:
        # Fallback (v12): near-identical lower-layer WavLM embeddings.
        print("ECAPA gives no plausible grouping; falling back to WavLM layers 1-4")
        E = np.stack([aemb_layers_train[f][1:5].astype(np.float32).mean(0) for f in train.filename])
        E = (E - E.mean(0)) / (E.std(0) + 1e-6)
        E /= np.linalg.norm(E, axis=1, keepdims=True)
        S = E @ E.T
        np.fill_diagonal(S, -1)
        _, SPEAKER_GROUPS = connected_components(S > 0.9, directed=False)
    else:
        print(f"chosen threshold: {CHOSEN}")
    yl = lab.loc[train.filename].values
    same = SPEAKER_GROUPS[:, None] == SPEAKER_GROUPS[None, :]
    np.fill_diagonal(same, False)
    i, j = np.nonzero(np.triu(same))
    sizes = np.bincount(SPEAKER_GROUPS)
    print(f"{len(sizes)} speaker groups for {len(train)} clips | {(sizes > 1).sum()} groups have 2+ clips | "
          f"largest group: {sizes.max()} clips; folds keep each group together")
    if len(i):
        # Same-speaker clips should get similar scores; random pairs differ by about sqrt(2) x the label std.
        print(f"label RMSE within groups {np.sqrt(((yl[i] - yl[j]) ** 2).mean()):.3f} vs random pairs "
              f"{np.sqrt(2) * yl.std():.3f}")

In [ ]:
y = crops.label.values  # one row per training crop (section 3b); every crop keeps its clip's score
PARENT = crops.parent.values
strata = (np.clip(train.label.values, 2, 5) * 2).astype(int)
def make_folds(seed):
    """Stratified 5-fold split of the clips. Repeat speakers stay in the same fold, and every crop follows its clip."""
    if SPEAKER_GROUPS is not None and SPEAKER_GROUPS.max() + 1 < len(train):
        clip_folds = StratifiedGroupKFold(5, shuffle=True, random_state=seed).split(train, strata, SPEAKER_GROUPS)
    else:
        clip_folds = StratifiedKFold(5, shuffle=True, random_state=seed).split(train, strata)
    return [(np.flatnonzero(np.isin(PARENT, tr)), np.flatnonzero(np.isin(PARENT, va))) for tr, va in clip_folds]


FOLDS = make_folds(SEED)
EMB_PREFIXES = ("temb_", "aemb_", "aembL_") + tuple(ENCODERS.values())
EMB_PCA = 32  # tuned in section 6b
EMB_REDUCE = "pca"  # "pca" or "none", tuned in section 6b


def preprocessor(cols, reduce=None):
    """Impute + scale hand-made features; scale (+ PCA) each embedding block."""
    reduce = reduce or EMB_REDUCE
    hand = [c for c in cols if not c.startswith(EMB_PREFIXES)]
    # Scaled hand-made features are clipped to +-5 so that a single extreme clip cannot dominate a linear model.
    clip5 = FunctionTransformer(np.clip, kw_args={"a_min": -5, "a_max": 5})
    blocks = [("hand", make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clip5), hand)] if hand else []
    for prefix in EMB_PREFIXES:
        emb = [c for c in cols if c.startswith(prefix)]
        if emb:
            steps = [StandardScaler()] + ([PCA(EMB_PCA, random_state=SEED)] if reduce == "pca" else [])
            blocks.append((prefix, make_pipeline(*steps), emb))
    return ColumnTransformer(blocks)


MODELS = {
    # cv=5 (contiguous folds) instead of leave-one-out: the 4 crops of a clip sit next to each other, so
    # they stay together. Leave-one-out would let a crop's siblings vouch for it and pick too little regularisation.
    "ridge": RidgeCV(alphas=np.logspace(-2, 4, 13), cv=5),
    "svr": SVR(C=2.0, epsilon=0.2),
    "lgbm": LGBMRegressor(n_estimators=500, learning_rate=0.02, num_leaves=15, min_child_samples=15 * N_CROPS,
                          subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,
                          random_state=SEED, verbose=-1),
}

def rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))


def pearson(a, b):
    return float(np.corrcoef(a, b)[0, 1])


def cv_predict(model, frame):
    """Out-of-fold predictions: each clip is predicted by a model that never saw it."""
    return cross_val_predict(make_pipeline(preprocessor(list(frame.columns)), clone(model)), frame, y, cv=FOLDS)


def oof(model, cols):
    return cv_predict(model, X[cols])

### 6b. Choosing encoder layers and embedding size

In speech encoders, lower layers describe sound and voice, middle layers describe pronunciation and fluency, and the top layers are specialised for the model's own training task. Instead of guessing, for WavLM and each Whisper encoder every quarter of the layer stack (plus the middle half) is scored with Ridge on the same CV folds, and the best range is kept. The number of PCA components per embedding block is chosen the same way. Only a handful of options are tried, so the risk of overfitting the CV is small.

In [ ]:
def pooled_layers(layers, lo, hi, files, prefix):
    return pd.DataFrame(np.stack([layers[f][lo:hi].astype(np.float32).mean(0) for f in files]),
                        index=list(files)).add_prefix(prefix)


def best_layers(layers_train, layers_test, prefix, label):
    """Score each quarter of the layer stack (and the middle half) with Ridge; return the best pooled block."""
    n = next(iter(layers_train.values())).shape[0]
    q = n // 4
    ranges = [(0, q), (q, 2 * q), (2 * q, 3 * q), (3 * q, n), (q, 3 * q)]
    scores = {r: rmse(y, cv_predict(MODELS["ridge"], pooled_layers(layers_train, *r, crops.filename, prefix)))
              for r in ranges}
    lo, hi = min(scores, key=scores.get)
    print(f"{label:>18}: " + "  ".join(f"layers {a}-{b - 1}: {v:.4f}" for (a, b), v in scores.items())
          + f"  -> keep {lo}-{hi - 1}")
    return pooled_layers(layers_train, lo, hi, crops.filename, prefix), pooled_layers(layers_test, lo, hi, test.filename, prefix)


parts_train, parts_test = [ac_crops], [ac_test]
if RUN_DEEP:
    blocks = [best_layers(aemb_layers_crops, aemb_layers_test, "aemb_", "WavLM"),
              best_layers(aembL_layers_crops, aembL_layers_test, "aembL_", "WavLM-large")]
    blocks += [best_layers(*WHISPER_LAYERS[name], prefix, f"Whisper {name}") for name, prefix in ENCODERS.items()]
    parts_train += [fl_crops, gr_crops, er_crops, pos_crops, temb_crops] + [b[0] for b in blocks]
    parts_test += [fl_test, gr_test, er_test, pos_test, temb_test] + [b[1] for b in blocks]
LENGTH_FEATURES = ["ac_duration"]  # counts that depend on clip length; test clips are shorter
X = pd.concat(parts_train, axis=1).loc[crops.filename].astype(float).drop(columns=LENGTH_FEATURES)
X_test = pd.concat(parts_test, axis=1).loc[test.filename].astype(float).drop(columns=LENGTH_FEATURES)
ALL = list(X.columns)
GROUPS = {name: [c for c in ALL if c.startswith(prefix)] for name, prefix in
          [("audio quality", "ac_"), ("fluency", "fl_"), ("grammar", "gr_"), ("ERRANT error types", "er_"), ("grammar change (POS tags)", "td_"), ("grammar complexity", "cx_"), ("text embedding", "temb_"),
           ("WavLM embedding", "aemb_"), ("WavLM-large embedding (mean + std)", "aembL_")] + [(f"Whisper {n} encoder", pre) for n, pre in ENCODERS.items()]}
GROUPS = {k: v for k, v in GROUPS.items() if v}
print("X:", X.shape, "| X_test:", X_test.shape, "|", {k: len(v) for k, v in GROUPS.items()})

if RUN_DEEP:
    pca_scores = {}
    for k in [64, 128, 256]:
        EMB_PCA = k
        pca_scores[k] = rmse(y, oof(MODELS["ridge"], ALL))
    EMB_PCA = min(pca_scores, key=pca_scores.get)
    print("Ridge CV RMSE by PCA size:", {k: round(v, 4) for k, v in pca_scores.items()}, "| chosen:", EMB_PCA)
    EMB_REDUCE = "none"
    no_pca = rmse(y, oof(MODELS["ridge"], ALL))
    EMB_REDUCE = "none" if no_pca < pca_scores[EMB_PCA] else "pca"
    print(f"Ridge CV RMSE without PCA: {no_pca:.4f} | chosen reduction: {EMB_REDUCE}")

### 6b-2. How each hand-made feature relates to the human score

Spearman rank correlation of every hand-made feature with the score, on the training clips. Features with |r| < 0.1 carry almost no signal on their own (grey bars). A weak feature can still help in combination with others, so the weak ones are dropped only if Ridge cross-validation does not get worse without them. The right panel shows the clip-length bias: longer training clips score slightly higher, while test clips are shorter. That is why duration was removed from the features.

In [ ]:
hand_all = [c for c in ALL if not c.startswith(EMB_PREFIXES)]
score = pd.Series(y, index=X.index)
corr = X[hand_all].corrwith(score, method="spearman").sort_values()
colors = np.where(corr.abs() < 0.1, "lightgray", np.where(corr > 0, "tab:green", "tab:red"))

fig, ax = plt.subplots(1, 2, figsize=(17, max(5, 0.24 * len(corr))), gridspec_kw={"width_ratios": [3, 2]})
corr.plot.barh(ax=ax[0], color=colors)
for v in (-0.1, 0.1):
    ax[0].axvline(v, ls="--", c="k", lw=0.8)
ax[0].set(xlabel="Spearman r with human score", title="Hand-made features vs human score (grey = |r| < 0.1)")
dur_tr = ac_train.loc[train.filename].ac_duration.values
ax[1].scatter(dur_tr, train.label + np.random.uniform(-0.1, 0.1, len(train)), s=8, alpha=0.4, label="train clips")
ax[1].axvline(dur_tr.mean(), c="tab:blue", ls="--", label=f"train mean {dur_tr.mean():.1f} s")
ax[1].axvline(ac_test.ac_duration.mean(), c="tab:orange", ls="--", label=f"test mean {ac_test.ac_duration.mean():.1f} s")
ax[1].set(xlabel="clip duration (s)", ylabel="human score (jittered)",
          title=f"Length bias: Spearman r = {pd.Series(dur_tr).corr(train.label, method='spearman'):.3f}")
ax[1].legend()
plt.tight_layout()
plt.show()

# The POS groups (section 4d) stay only if Ridge CV improves with them.
NEW = [c for c in ALL if c.startswith(("td_", "cx_"))]
if NEW:
    base = [c for c in ALL if c not in NEW]
    r_with, r_without = rmse(y, oof(MODELS["ridge"], ALL)), rmse(y, oof(MODELS["ridge"], base))
    print(f"POS tag-change + complexity ({len(NEW)} features): Ridge CV RMSE without {r_without:.4f} | "
          f"with {r_with:.4f} | change {r_with - r_without:+.4f}")
    if r_with >= r_without:
        X, X_test, ALL = X[base], X_test[base], base
        GROUPS = {k: v for k, v in GROUPS.items() if not v[0].startswith(("td_", "cx_"))}
        print("-> dropped: no CV gain")
    else:
        print("-> kept")
    corr = corr[corr.index.isin(ALL)]

WEAK = corr[corr.abs() < 0.1].index.tolist()
if WEAK:
    keep = [c for c in ALL if c not in WEAK]
    r_all, r_keep = rmse(y, oof(MODELS["ridge"], ALL)), rmse(y, oof(MODELS["ridge"], keep))
    print(f"{len(WEAK)} weak features: {WEAK}")
    print(f"Ridge CV RMSE with them {r_all:.4f} | without them {r_keep:.4f}")
    if r_keep <= r_all:
        X, X_test, ALL = X[keep], X_test[keep], keep
        GROUPS = {k: [c for c in v if c in ALL] for k, v in GROUPS.items()}
        GROUPS = {k: v for k, v in GROUPS.items() if v}
        print("-> dropped; X:", X.shape)
    else:
        print("-> kept: they still help in combination")

### 6b-3. A text model fine-tuned on the transcripts

The test speakers are new, so the models must rely on what generalises across speakers. A voice embedding partly describes *who* is speaking; the words describe *how well they use grammar*. So far the transcript is used only through hand-made counts and a frozen sentence embedding (CV RMSE about 0.77 on its own).

Here the transcripts come from Whisper `large-v3-turbo`, which recognises speech much better than `small.en` and is prompted the same way to keep fillers and false starts. Two DeBERTa-v3 models (base and large) are fine-tuned to predict the score directly from the transcript of each training crop. The large one has 3 times as many parameters and is trained with a smaller learning rate and batch size.
- It is trained inside the same speaker-grouped folds, so every out-of-fold prediction comes from a model that never saw that speaker.
- Each fold's model also predicts the test clips, and the 5 test predictions are averaged.
- Their out-of-fold predictions join the stacked ensemble as two more base models (section 6c).

In [ ]:
TEXT_MODELS = {  # name -> (checkpoint, learning rate, batch size)
    "DeBERTa on transcripts": ("microsoft/deberta-v3-base", 3e-5, 16),
    "DeBERTa-large on transcripts": ("microsoft/deberta-v3-large", 1e-5, 8),
}
ASR_TEXT = "turbo"  # Whisper large-v3-turbo


def finetune_text(checkpoint, train_texts, train_y, pred_texts, epochs=3, lr=3e-5, bs=16, max_len=256):
    """Fine-tune a DeBERTa regressor on (transcript, score); return its predictions for each list in pred_texts."""
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
    # DeBERTa's TorchScript position helpers crash this torch build's CUDA fuser ("fabs" overload); run them unfused.
    torch._C._jit_set_texpr_fuser_enabled(False)
    torch._C._jit_override_can_fuse_on_gpu(False)
    torch.manual_seed(SEED)
    tok = AutoTokenizer.from_pretrained(checkpoint)
    # fp32 master weights (newer transformers keep the checkpoint's fp16); autocast still runs the maths in fp16.
    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=1).float().to(DEVICE)
    mu, sd = train_y.mean(), train_y.std()
    target = torch.tensor((train_y - mu) / sd, dtype=torch.float32)

    def encode(texts):
        return tok(list(texts), truncation=True, max_length=max_len, padding=True, return_tensors="pt").to(DEVICE)

    n_batches = int(np.ceil(len(train_texts) / bs))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, int(0.1 * epochs * n_batches), epochs * n_batches)
    scaler = torch.cuda.amp.GradScaler()
    rng = np.random.default_rng(SEED)
    model.train()
    for _ in range(epochs):
        for idx in np.array_split(rng.permutation(len(train_texts)), n_batches):
            with torch.autocast("cuda", dtype=torch.float16):
                out = model(**encode(train_texts[idx])).logits.reshape(-1)
            loss = torch.nn.functional.mse_loss(out.float(), target[idx].to(DEVICE))
            opt.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
    model.eval()
    preds = []
    with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
        for texts in pred_texts:
            out = [model(**encode(texts[i:i + 64])).logits.reshape(-1).float().cpu().numpy() for i in range(0, len(texts), 64)]
            preds.append(np.concatenate(out) * sd + mu)
    del model, opt
    torch.cuda.empty_cache()
    return preds


def text_model_oof(checkpoint, lr, bs):
    """Out-of-fold predictions on the training crops, plus averaged predictions for all training crops and the test set."""
    oof, train_p, test_p = np.zeros(len(crops)), [], []
    for k, (tr, va) in enumerate(FOLDS):
        p_va, p_tr, p_te = finetune_text(checkpoint, crop_texts[tr], y[tr], [crop_texts[va], crop_texts, test_texts], lr=lr, bs=bs)
        oof[va] = p_va
        train_p.append(p_tr)
        test_p.append(p_te)
        print(f"fold {k}: CV RMSE {rmse(y[va], p_va):.4f}")
    return oof, np.mean(train_p, 0), np.mean(test_p, 0)


EXTRA = {}  # base models trained outside sklearn: name -> (out-of-fold, train, test) predictions
if RUN_DEEP:
    done = all((WORK_DIR / f"asr_{split}_{ASR_TEXT}.pkl").exists() for split in ("train", "test"))
    asr_big = None if done else whisper.load_model(ASR_TEXT, device=DEVICE)
    asr_train_big = cached(f"asr_train_{ASR_TEXT}",
                           lambda: {f: transcribe(asr_big, p) for f, p in tqdm(list(zip(train.filename, train.path)))})
    asr_test_big = cached(f"asr_test_{ASR_TEXT}",
                          lambda: {f: transcribe(asr_big, p) for f, p in tqdm(list(zip(test.filename, test.path)))})
    del asr_big
    torch.cuda.empty_cache()
    crop_texts = np.array([crop_asr(asr_train_big[f], t0, t1)["text"] for f, t0, t1 in
                           zip(train.filename.values[crops.parent], crops.start, crops.end)])
    test_texts = np.array([asr_test_big[f]["text"] for f in test.filename])
    fold_key = hashlib.md5(np.concatenate([va for _, va in FOLDS]).tobytes()).hexdigest()[:8]
    # Free the GPU for fine-tuning: earlier sections leave their models loaded, and DeBERTa-large needs ~10 GB.
    import gc, sys
    sys.last_type = sys.last_value = sys.last_traceback = sys.last_exc = None  # frames of an earlier error
    for v in list(globals().values()):
        if isinstance(v, torch.nn.Module):
            v.cpu()
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU memory in use before fine-tuning: {torch.cuda.memory_allocated() / 2**30:.1f} GiB")
    for name, (ckpt, lr, bs) in TEXT_MODELS.items():
        EXTRA[name] = cached(f"deberta_{ckpt.split('/')[-1]}_{ASR_TEXT}_crops{N_CROPS}_{fold_key}",
                             lambda: text_model_oof(ckpt, lr, bs))
        o = EXTRA[name][0]
        print(f"{name}: CV RMSE {rmse(y, o):.4f} | Pearson {pearson(y, o):.4f}")

### 6c. A diverse ensemble

An ensemble only helps when its members make *different* mistakes. Ridge, SVR and LightGBM all read the same feature table, so they make similar mistakes. More variety comes from:
- **one Ridge per feature group:** each embedding block on its own, plus one for the hand-made features. The stacker then learns how much to trust each source.
- **different model families:** kernel ridge with an RBF kernel, and k-nearest neighbours on the Whisper `medium.en` embedding.
- **bagging over fold splits:** every base model's out-of-fold prediction is averaged over 3 different shuffles of the 5 folds. This makes each prediction steadier without leaking labels, because in every split a clip is predicted by a model that never saw it.

With about a dozen base models, a plain stacker can overfit. Two non-negative stackers are compared on the same folds, linear regression and ridge regression, and the better one is kept.

In [ ]:
from sklearn.compose import TransformedTargetRegressor
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor

hand_cols = [c for c in ALL if not c.startswith(EMB_PREFIXES)]
BASE_MODELS = {n: (m, ALL) for n, m in MODELS.items()}  # name -> (model, feature columns)
# KernelRidge has no intercept, so the target is standardised around it.
BASE_MODELS["kernel ridge (RBF)"] = (TransformedTargetRegressor(KernelRidge(alpha=1.0, kernel="rbf"), transformer=StandardScaler()), ALL)
BASE_MODELS["ridge: hand-made features"] = (MODELS["ridge"], hand_cols)
for g, cols in GROUPS.items():
    if cols[0].startswith(EMB_PREFIXES):
        BASE_MODELS[f"ridge: {g}"] = (MODELS["ridge"], cols)
if "Whisper medium.en encoder" in GROUPS:
    BASE_MODELS["kNN: Whisper medium.en"] = (KNeighborsRegressor(20 * N_CROPS), GROUPS["Whisper medium.en encoder"])

SPLITS = [FOLDS] + [make_folds(SEED + r) for r in (1, 2)]


def make_pipe(name, model, cols):
    """LightGBM always gets PCA-reduced embeddings; the other models use the reduction chosen in 6b."""
    return make_pipeline(preprocessor(cols, "pca" if name == "lgbm" else None), clone(model))


def oof_bagged(name, model, cols):
    """Out-of-fold predictions averaged over several fold splits."""
    pipe = make_pipe(name, model, cols)
    return np.mean([cross_val_predict(clone(pipe), X[cols], y, cv=f) for f in SPLITS], axis=0)


OOF = {}
for n, (m, cols) in BASE_MODELS.items():
    OOF[n] = oof_bagged(n, m, cols)
    print(f"{n:>34}: CV RMSE {rmse(y, OOF[n]):.4f} | Pearson {pearson(y, OOF[n]):.4f}")
for n, (o, _, _) in EXTRA.items():
    OOF[n] = o
BASE = list(OOF)  # the base models fed to the stacker
P = np.column_stack([OOF[n] for n in BASE])
OOF["simple average"] = np.clip(P.mean(1), 1, 5)

# Stacking: a non-negative linear model (with intercept) on the out-of-fold predictions. Its weights
# may sum to more than 1, which stretches the squeezed predictions back out. Its own CV uses FOLDS.
STACKERS = {"linear": LinearRegression(positive=True), "ridge": Ridge(alpha=1.0, positive=True)}
stack_cv = {k: np.clip(cross_val_predict(m, P, y, cv=FOLDS), 1, 5) for k, m in STACKERS.items()}
best = min(stack_cv, key=lambda k: rmse(y, stack_cv[k]))
print("stacker CV RMSE:", {k: round(rmse(y, v), 4) for k, v in stack_cv.items()}, "| chosen:", best)
stacker = STACKERS[best].fit(P, y)
OOF["ensemble"] = stack_cv[best]
weights = pd.Series(stacker.coef_, BASE).round(3)
print(weights[weights > 0].sort_values(ascending=False).to_string(),
      f"\nsum {stacker.coef_.sum():.3f} | intercept {stacker.intercept_:.3f}")

## 7. Results

**Training RMSE** comes from models fitted on all training crops and scored on those same crops; this number is required by the brief. **CV** numbers come from out-of-fold predictions on single test-length crops, which is what a test clip looks like, so they estimate test performance. The *mean baseline* always predicts the average training score.

In [ ]:
FITTED = {n: make_pipe(n, m, cols).fit(X[cols], y) for n, (m, cols) in BASE_MODELS.items()}
train_pred = {n: FITTED[n].predict(X[BASE_MODELS[n][1]]) for n in BASE_MODELS}
train_pred.update({n: v[1] for n, v in EXTRA.items()})
P_train = np.column_stack([train_pred[n] for n in BASE])
train_pred["simple average"] = np.clip(P_train.mean(1), 1, 5)
train_pred["ensemble"] = np.clip(stacker.predict(P_train), 1, 5)
OOF["mean baseline"] = np.full(len(y), y.mean())
train_pred["mean baseline"] = OOF["mean baseline"]

results = pd.DataFrame({n: {"train RMSE": rmse(y, train_pred[n]), "CV RMSE": rmse(y, OOF[n]),
                            "CV Pearson": pearson(y, OOF[n]) if n != "mean baseline" else np.nan,
                            "CV MAE": float(np.mean(np.abs(y - OOF[n])))} for n in OOF}).T.round(4)
print(f"TRAINING RMSE (ensemble): {results.loc['ensemble', 'train RMSE']:.4f}")
clip_cv = pd.Series(OOF["ensemble"]).groupby(PARENT).mean().values
print(f"ensemble CV RMSE per test-length crop: {results.loc['ensemble', 'CV RMSE']:.4f} | "
      f"averaged over each clip's {N_CROPS} crops: {rmse(train.label.values, clip_cv):.4f}")
results

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ens = OOF["ensemble"]
ax[0].scatter(y + np.random.uniform(-0.1, 0.1, len(y)), ens, s=10, alpha=0.5)
ax[0].plot([1, 5], [1, 5], "k--")
ax[0].set(xlabel="true score (jittered)", ylabel="out-of-fold prediction",
          title=f"Ensemble: CV RMSE {rmse(y, ens):.3f}, r = {pearson(y, ens):.3f}")

levels = sorted(set(y))
ax[1].boxplot([ens[y == lv] - lv for lv in levels])
ax[1].set_xticks(range(1, len(levels) + 1), levels)
ax[1].axhline(0, c="k", ls="--")
ax[1].set(xlabel="true score", ylabel="prediction - truth", title="Errors by score level (shrinkage toward the mean)")

hand = [c for c in ALL if not c.startswith(EMB_PREFIXES)]
imp = clone(MODELS["lgbm"]).set_params(importance_type="gain").fit(X[hand], y)
pd.Series(imp.feature_importances_, hand).nlargest(15).sort_values().plot.barh(
    ax=ax[2], title="LightGBM gain importance (hand-made features)")
plt.tight_layout()
plt.show()

## 8. Ablation: what each feature group adds (Ridge, same folds)

In [ ]:
ablation = {g: rmse(y, oof(MODELS["ridge"], cols)) for g, cols in GROUPS.items()}
ablation["all features"] = rmse(y, OOF["ridge"])
ablation = pd.Series(ablation).sort_values().round(4).rename("Ridge CV RMSE")
ablation.plot.barh(figsize=(7, 3), title="Lower is better")
plt.show()
ablation.to_frame()

## 9. Test predictions and submission

The final models are fitted on all training clips and combined by the stacker. Predictions are clipped to [1, 5]. A noise-only clip would get 0, matching the training labels; this guard never fires on the current test set.

In [ ]:
test_pred = np.clip(stacker.predict(np.column_stack([FITTED[n].predict(X_test[BASE_MODELS[n][1]]) if n in FITTED
                                                      else EXTRA[n][2] for n in BASE])), 1, 5)
test_pred[is_noise(ac_test.loc[test.filename]).values] = 0.0
sub = pd.DataFrame({"filename": test.filename, "label": test_pred.round(4)})
sub.to_csv(WORK_DIR / "submission.csv", index=False)
assert len(sub) == 216 and sub.label.between(0, 5).all()
print(sub.describe().T)
from IPython.display import FileLink
FileLink("submission.csv")  # download link

## 10. Conclusions and next steps

- **Data problems mattered more than model size.** Removing the noise clips, training on test-length crops and keeping each speaker inside one CV fold made the validation honest and the leaderboard score better.
- **What is said matters as much as how it sounds.** Frozen speech-model embeddings (Whisper and WavLM encoders) are the strongest single feature groups (section 8), but the biggest leaderboard gain came from fine-tuning DeBERTa on good transcripts (0.390 → 0.350). The test speakers are new, and the words describe grammar without describing the voice.
- **Diversity beats more of the same.** The stacker gives most weight to SVR on all features and to the DeBERTa models, which make different mistakes. More text models of the same kind, or fine-tuning a larger speech model, gave no further gain.
- **Predictions still shrink toward the mean.** Clips scored 2.0 are predicted too high and clips scored 4.5–5.0 too low; only 4 training clips score below 2.0, too few to learn the low end.
- **Next steps:** a transcription model that keeps disfluencies verbatim (Whisper drops most fillers), and averaging several fine-tuning seeds of the text models to make them steadier.